# 1.Import Libraries

In [1]:
import os
import cv2
import torch
import timm
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

from facenet_pytorch import MTCNN

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


# 2.Define Paths

In [3]:
video_folder = "/mnt/d/deepfake/deepfake-env/FF++"
output_folder = "/mnt/d/deepfake/tf-gpu-env/complete_project/temporal_frames"
os.makedirs(output_folder, exist_ok=True)

frames_per_video = 15
frames_per_segment = 3

# 3.Frame Extraction

In [4]:
classes = ["real", "fake"]

for cls in classes:
    class_video_folder = os.path.join(video_folder, cls)
    class_output_folder = os.path.join(output_folder, cls)
    os.makedirs(class_output_folder, exist_ok=True)
    video_files = os.listdir(class_video_folder)
    
    for video_file in tqdm(video_files, desc=f"Processing {cls}"):
        if not video_file.endswith((".mp4", ".avi", ".mov")):
            continue
        video_path = os.path.join(class_video_folder, video_file)
        video_name = os.path.splitext(video_file)[0]
        save_folder = os.path.join(class_output_folder, video_name)
        os.makedirs(save_folder, exist_ok=True)
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        segments = frames_per_video // frames_per_segment
        segment_size = total_frames // segments
        saved_count = 0
        
        for seg in range(segments):
            start_frame = seg * segment_size + segment_size // 2
            for i in range(frames_per_segment):
                frame_id = start_frame + i
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
                ret, frame = cap.read()
                if not ret:
                    break
                frame_name = os.path.join(save_folder, f"frame_{saved_count}.jpg")
                cv2.imwrite(frame_name, frame)
                saved_count += 1
        cap.release()
print("Frame extraction completed.")

Processing fake: 100%|████████████████████████████████████████████████████████████████| 200/200 [13:10<00:00,  3.95s/it]

Frame extraction completed.


# 3. Paths(Faces)

In [6]:
frames_folder = "/mnt/d/deepfake/tf-gpu-env/complete_project/temporal_frames"

faces_folder = "/mnt/d/deepfake/tf-gpu-env/complete_project/temporal_faces"

os.makedirs(faces_folder, exist_ok=True)

# 4.Face Detector

In [162]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

mtcnn = MTCNN(
    image_size=224,
    margin=20,
    device=device
)

print("Face detector ready on:", device)

Face detector ready on: cuda


# 5.Face Extraction

In [10]:
classes = ["real", "fake"]

for cls in classes:
    class_frame_path = os.path.join(frames_folder, cls)
    class_face_path = os.path.join(faces_folder, cls)
    os.makedirs(class_face_path, exist_ok=True)

    videos = os.listdir(class_frame_path)

    for video in tqdm(videos, desc=f"Processing {cls}"):
        video_frame_folder = os.path.join(class_frame_path, video)
        video_face_folder = os.path.join(class_face_path, video)
        os.makedirs(video_face_folder, exist_ok=True)

        frames = sorted(os.listdir(video_frame_folder))

        for frame in frames:
            frame_path = os.path.join(video_frame_folder, frame)
            img = cv2.imread(frame_path)

            if img is None:
                continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            face = mtcnn(img_rgb)

            if face is not None:
                face = face.permute(1, 2, 0).cpu().numpy()

                # Convert from [-1,1] → [0,255]
                face = ((face + 1) / 2 * 255).clip(0,255).astype("uint8")

                face_path = os.path.join(video_face_folder, frame)

                cv2.imwrite(face_path, cv2.cvtColor(face, cv2.COLOR_RGB2BGR))

print("Face extraction completed.")

Processing fake: 100%|████████████████████████████████████████████████████████████████| 200/200 [07:27<00:00,  2.24s/it]

Face extraction completed.


In [69]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# 6.Train test split

In [3]:
import os
import shutil
import random
from tqdm import tqdm

# original dataset
source_dataset = "/mnt/d/deepfake/tf-gpu-env/complete_project/temporal_faces"

# new split dataset
target_dataset = "/mnt/d/deepfake/tf-gpu-env/complete_project/temporal_split"

classes = ["real", "fake"]

split_ratio = 0.8   # 80% train, 20% test

for cls in classes:

    source_class = os.path.join(source_dataset, cls)

    videos = os.listdir(source_class)

    random.shuffle(videos)

    split_index = int(len(videos) * split_ratio)

    train_videos = videos[:split_index]
    test_videos = videos[split_index:]

    train_dir = os.path.join(target_dataset, "train", cls)
    test_dir = os.path.join(target_dataset, "test", cls)

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    # copy training videos
    for vid in tqdm(train_videos, desc=f"Copying TRAIN {cls}"):

        shutil.copytree(
            os.path.join(source_class, vid),
            os.path.join(train_dir, vid)
        )

    # copy test videos
    for vid in tqdm(test_videos, desc=f"Copying TEST {cls}"):

        shutil.copytree(
            os.path.join(source_class, vid),
            os.path.join(test_dir, vid)
        )

print("✅ Train/Test split created in temporal_split folder")

Copying TEST fake: 100%|██████████████████████████████████████████████████████████████| 158/158 [01:10<00:00,  2.25it/s]

✅ Train/Test split created in temporal_split folder


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [3]:
class TemporalDataset(Dataset):
    
    def __init__(self, root_dir, frames_per_video=15):
        self.root_dir = root_dir
        self.frames_per_video = frames_per_video
        self.samples = []
        classes = {"real":0, "fake":1}
        
        for cls in classes:
            class_path = os.path.join(root_dir, cls)
            
            for video in os.listdir(class_path):
                video_path = os.path.join(class_path, video)
                self.samples.append((video_path, classes[cls]))
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        frames = sorted(os.listdir(video_path))
        interval = max(len(frames)//self.frames_per_video,1)
        frames = frames[::interval][:self.frames_per_video]
        images = []

        for frame in frames:
            img_path = os.path.join(video_path, frame)
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.resize(img,(224,224))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = torch.tensor(img).permute(2,0,1).float()/255
            images.append(img)
            
        while len(images) < self.frames_per_video:
            images.append(images[-1])
        images = torch.stack(images)
        return images, label

In [4]:
train_dataset = TemporalDataset(
"/home/pratyush/deepfake_project/temporal_split/train",
frames_per_video=15
)

test_dataset = TemporalDataset(
"/home/pratyush/deepfake_project/temporal_split/test",
frames_per_video=15
)

train_loader = DataLoader(
train_dataset,
batch_size=4,
shuffle=True,
num_workers=4,
pin_memory=True
)

test_loader = DataLoader(
test_dataset,
batch_size=4,
shuffle=False,
num_workers=4
)

In [5]:
import timm
import torch.nn as nn

class EfficientNetLSTM(nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            num_classes=0
        )

        self.lstm = nn.LSTM(
            input_size=1280,
            hidden_size=256,
            batch_first=True
        )

        self.fc = nn.Linear(256,2)

    def forward(self,x):

        b,t,c,h,w = x.shape

        x = x.view(b*t,c,h,w)

        features = self.backbone(x)

        features = features.view(b,t,-1)

        lstm_out,_ = self.lstm(features)

        out = lstm_out[:,-1,:]

        return self.fc(out)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EfficientNetLSTM().to(device)
print("Model loaded on:", device)

Model loaded on: cuda


In [7]:
criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
model.parameters(),
lr=1e-4
)

num_epochs = 10

best_acc = 0

save_path = "/mnt/d/deepfake/tf-gpu-env/complete_project/models/efficientnet_lstm_best.pth"

In [18]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for frames, labels in loop:
        frames = frames.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        acc = 100 * correct / total
        loop.set_postfix(loss=loss.item(), acc=f"{acc:.2f}%")
    epoch_acc = 100 * correct / total
    print("Epoch accuracy:", epoch_acc)

    if epoch_acc > best_acc:
        best_acc = epoch_acc
        torch.save(model.state_dict(), save_path)
        print("✅ Best model saved")

Epoch 1/10: 100%|█████████████████████████████████████████████| 316/316 [40:04<00:00,  7.61s/it, acc=70.09%, loss=0.125]


Epoch accuracy: 70.09493670886076
✅ Best model saved


Epoch 2/10: 100%|██████████████████████████████████████████████| 316/316 [39:07<00:00,  7.43s/it, acc=85.52%, loss=1.24]


Epoch accuracy: 85.52215189873418
✅ Best model saved


Epoch 3/10: 100%|█████████████████████████████████████████████| 316/316 [37:58<00:00,  7.21s/it, acc=89.87%, loss=0.214]


...

Epoch accuracy: 97.15189873417721


Epoch 9/10: 100%|████████████████████████████████████████████| 316/316 [38:03<00:00,  7.23s/it, acc=98.02%, loss=0.0028]


Epoch accuracy: 98.02215189873418


Epoch 10/10: 100%|█████████████████████████████████████████████| 316/316 [37:31<00:00,  7.12s/it, acc=98.50%, loss=0.11]


Epoch accuracy: 98.49683544303798
✅ Best model saved


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EfficientNetLSTM()

model.load_state_dict(torch.load(
 "/mnt/d/deepfake/tf-gpu-env/complete_project/models/efficientnet_lstm_best.pth"
))

model = model.to(device)

model.eval()

print("Model loaded successfully")

Model loaded successfully


In [9]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

all_preds = []
all_labels = []

with torch.no_grad():

    for frames, labels in test_loader:

        frames = frames.to(device)

        outputs = model(frames)

        _, predicted = torch.max(outputs,1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

Confusion Matrix:
[[146  12]
 [ 29 129]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.92      0.88       158
           1       0.91      0.82      0.86       158

    accuracy                           0.87       316
   macro avg       0.87      0.87      0.87       316
weighted avg       0.87      0.87      0.87       316



In [10]:
from facenet_pytorch import MTCNN

mtcnn = MTCNN(
    image_size=224,
    margin=40,
    device=device
)

In [43]:
def predict_video(video_path):

    cap = cv2.VideoCapture(video_path)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    frames_needed = 15
    sampling_frames = 60

    interval = max(total_frames // sampling_frames, 1)

    frames = []

    for i in range(sampling_frames):

        frame_id = i * interval
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)

        ret, frame = cap.read()

        if not ret:
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        face = mtcnn(frame_rgb)

        if face is not None:
            face = face.permute(1,2,0).cpu().numpy()
        else:
            h, w, _ = frame_rgb.shape
            face = frame_rgb[h//4:3*h//4, w//4:3*w//4]

        face = torch.tensor(face, dtype=torch.float32).permute(2,0,1) / 255.0

        frames.append(face)

    cap.release()

    print("Frames collected:", len(frames))

    if len(frames) == 0:
        print("❌ Could not extract frames")
        return

    sequences = []

    for i in range(0, len(frames)-frames_needed+1, 5):

        seq = frames[i:i+frames_needed]

        if len(seq) == frames_needed:
            sequences.append(torch.stack(seq))

    if len(sequences) == 0:

        while len(frames) < frames_needed:
            frames.append(frames[-1])

        sequences.append(torch.stack(frames[:frames_needed]))

    logits_list = []

    with torch.no_grad():

        for seq in sequences:

            seq = seq.unsqueeze(0).to(device)

            output = model(seq)

            logits_list.append(output.cpu())

    logits = torch.cat(logits_list, dim=0)

    avg_logits = logits.mean(dim=0)

    print("Scores:", avg_logits.numpy())

    # ⭐ Correct prediction using argmax
    pred_class = torch.argmax(avg_logits).item()

    print("Predicted class index:", pred_class)

    # adjust labels if needed
    if pred_class == 0:
        label = "FAKE"
    else:
        label = "REAL"

    print(f"Final Prediction: {label}")

    return label

In [48]:
predict_video("/mnt/c/Users/Pratyush/Downloads/balanced_celeb_df/fake/id56_id58_0009.mp4")

Frames collected: 60
Scores: [-0.24909453  0.28519374]
Predicted class index: 1
Final Prediction: REAL


'REAL'

In [17]:
import os

def predict_folder(folder_path):

    videos = [v for v in os.listdir(folder_path) if v.endswith(".mp4")]

    total = 0
    real_count = 0
    fake_count = 0

    for video in videos:

        video_path = os.path.join(folder_path, video)

        print("\n==============================")
        print("Testing:", video)

        result = predict_video(video_path)

        total += 1

        if result == "REAL":
            real_count += 1
        else:
            fake_count += 1

    print("\n========== SUMMARY ==========")
    print("Total videos:", total)
    print("Predicted REAL:", real_count)
    print("Predicted FAKE:", fake_count)

In [18]:
predict_folder("/mnt/c/Users/Pratyush/Downloads/balanced_celeb_df/real")


Testing: id0_0000.mp4
Frames collected: 60
Probabilities: [0.36873597 0.63124526]
Final Prediction: FAKE (63.12% confidence)

Testing: id0_0001.mp4
Frames collected: 60
Probabilities: [0.37051034 0.62928134]
Final Prediction: FAKE (62.93% confidence)

Testing: id0_0002.mp4
Frames collected: 60
Probabilities: [0.37043443 0.6295168 ]
Final Prediction: FAKE (62.95% confidence)

Testing: id0_0003.mp4
Frames collected: 60
Probabilities: [0.37122416 0.62850535]
Final Prediction: FAKE (62.85% confidence)

Testing: id0_0004.mp4
Frames collected: 60
Probabilities: [0.37439272 0.6255186 ]
Final Prediction: FAKE (62.55% confidence)

Testing: id0_0005.mp4
Frames collected: 60
Probabilities: [0.37390873 0.6253685 ]
Final Prediction: FAKE (62.54% confidence)

Testing: id0_0006.mp4
Frames collected: 60
Probabilities: [0.3689873  0.63060176]
Final Prediction: FAKE (63.06% confidence)

Testing: id0_0007.mp4
Frames collected: 60
Probabilities: [0.36974186 0.629901  ]
Final Prediction: FAKE (62.99% conf

KeyboardInterrupt: 